In [0]:
# ============================================================
# NOTEBOOK: nb_03_CustomerPortfolio
# PURPOSE:  Replaces SQL Procedure 3 (usp_LoadCustomerPortfolio)
#           Reads 7 source systems via metadata config, applies
#           exchange rates, enriches products/customers, applies
#           business rules, and MERGEs into warehouse.customer_portfolio.
# COMPLEXITY: HIGH (metadata-driven extraction + multi-layer enrichment)
# ============================================================

import uuid
from datetime import datetime
from delta.tables import DeltaTable
import pyspark.sql.functions as F

# --------------------------------------------------------
# PARAMETERS
# --------------------------------------------------------
dbutils.widgets.text("business_date", "2026-01-31", "Business Date")
dbutils.widgets.text("load_type", "FULL", "Load Type (FULL/INCREMENTAL)")
dbutils.widgets.text("debug", "1", "Debug Mode (1=print, 0=silent)")

business_date = dbutils.widgets.get("business_date")
load_type     = dbutils.widgets.get("load_type").upper()
debug         = int(dbutils.widgets.get("debug"))

if load_type not in ("FULL", "INCREMENTAL"):
    raise ValueError("Invalid Load Type. Must be FULL or INCREMENTAL.")

# --------------------------------------------------------
# AUDIT VARIABLES
# --------------------------------------------------------
execution_id   = str(uuid.uuid4())
procedure_name = "nb_03_CustomerPortfolio"
start_time     = datetime.now()

rows_read     = 0
rows_inserted = 0
rows_updated  = 0
rows_rejected = 0

if debug:
    print("=" * 50)
    print("NB_03_CUSTOMERPORTFOLIO STARTED")
    print("=" * 50)
    print(f"Execution ID : {execution_id}")
    print(f"Business Date: {business_date}")
    print(f"Load Type    : {load_type}")

NB_03_CUSTOMERPORTFOLIO STARTED
Execution ID : 271702c1-6499-48c3-8cc1-ea439ba17853
Business Date: 2026-01-31
Load Type    : FULL


In [0]:
# ============================================================
# AUDIT LOG FUNCTIONS
# Same pattern as nb_01 and nb_02.
# ============================================================

def write_audit_start():
    sql = f"""
        INSERT INTO retailbank_dev.audit.etl_execution_log 
        (execution_id, procedure_name, business_date, load_type, start_time, status)
        VALUES 
        ('{execution_id}', '{procedure_name}', '{business_date}', '{load_type}',
         '{start_time.strftime("%Y-%m-%d %H:%M:%S")}', 'RUNNING')
    """
    spark.sql(sql)


def write_audit_end(status, message):
    end_time         = datetime.now()
    duration_seconds = int((end_time - start_time).total_seconds())
    safe_message     = message.replace("'", "''")
    
    sql = f"""
        UPDATE retailbank_dev.audit.etl_execution_log
        SET 
            end_time         = '{end_time.strftime("%Y-%m-%d %H:%M:%S")}',
            status           = '{status}',
            rows_read        = {rows_read},
            rows_inserted    = {rows_inserted},
            rows_updated     = {rows_updated},
            rows_rejected    = {rows_rejected},
            duration_seconds = {duration_seconds},
            message          = '{safe_message}'
        WHERE execution_id = '{execution_id}'
    """
    spark.sql(sql)


write_audit_start()

In [0]:
# ============================================================
# READ CONFIGURATION & REFERENCE DATA
# 1. PortfolioSourceConfiguration -> tells us which tables/columns
# 2. ExchangeRates -> for currency conversion to USD
# 3. ProductReference -> for product descriptions
# 4. CustomerMaster -> for names, categories, branch codes
# 5. CustomerMasterExceptions -> to find customers with OPEN CRITICAL
# ============================================================

# 3.1 Portfolio Source Configuration
config_df = spark.table("retailbank_dev.config.portfolio_source_configuration") \
    .filter(F.col("is_active") == True) \
    .orderBy("load_priority")

if config_df.count() == 0:
    raise Exception("No active portfolio source systems configured.")

# 3.2 Exchange Rates (broadcast-ready small table)
fx_df = spark.table("retailbank_dev.reference.exchange_rates") \
    .filter(F.col("effective_date") == business_date)

if fx_df.count() == 0:
    raise Exception(f"Exchange rates not found for business date {business_date}.")

# 3.3 Product Reference
product_df = spark.table("retailbank_dev.reference.product") \
    .filter(F.col("product_status") == "ACTIVE")

# 3.4 Customer Master
customer_df = spark.table("retailbank_dev.warehouse.customer_master")

# 3.5 Critical Exceptions (for business rule exclusion)
critical_exceptions_df = spark.table("retailbank_dev.warehouse.customer_master_exceptions") \
    .filter((F.col("exception_status") == "OPEN") & (F.col("severity_code") == "CRITICAL")) \
    .select("customer_id").distinct()

if debug:
    print("Configuration loaded:")
    print(f"  Active sources : {config_df.count()}")
    print(f"  FX rates       : {fx_df.count()}")
    print(f"  Products       : {product_df.count()}")
    print(f"  Customers      : {customer_df.count()}")
    print(f"  Critical excep.: {critical_exceptions_df.count()}")
    config_df.select("source_system_code", "product_table", "customer_field", "load_priority").show(truncate=False)

Configuration loaded:
  Active sources : 7
  FX rates       : 5
  Products       : 10
  Customers      : 12
  Critical excep.: 0
+------------------+----------------------------+---------------+-------------+
|source_system_code|product_table               |customer_field |load_priority|
+------------------+----------------------------+---------------+-------------+
|CORE_BANKING      |source.core_banking_accounts|customer_number|1            |
|LOANS             |source.loan_accounts        |client_id      |2            |
|CARDS             |source.card_accounts        |client_number  |3            |
|INVESTMENTS       |source.investment_accounts  |investor_id    |4            |
|MORTGAGE          |source.mortgage_accounts    |client_code    |5            |
|MOBILE            |source.mobile_wallet        |customer_number|6            |
|FOREX             |source.forex_accounts       |customer_id    |7            |
+------------------+----------------------------+---------------+------

In [0]:
# ============================================================
# EXTRACTION PHASE
# 
# This replaces the SQL CURSOR. We loop over the config DataFrame,
# read each source table, and dynamically rename columns using
# the mappings stored in config.portfolio_source_configuration.
#
# Each source has different column names:
#   CORE_BANKING: customer_number, product_code, account_balance...
#   LOANS:        client_id, loan_product, outstanding_amount...
# We read the config to know which column is which.
# ============================================================

source_dataframes = []

for row in config_df.collect():
    source_code = row.source_system_code
    table_name  = row.product_table           # e.g. "source.core_banking_accounts"
    
    # Column mappings from config
    cust_col    = row.customer_field          # e.g. "customer_number"
    prod_col    = row.product_field           # e.g. "product_code"
    bal_col     = row.balance_field           # e.g. "account_balance"
    curr_col    = row.currency_field          # e.g. "currency"
    stat_col    = row.status_field            # e.g. "account_status"
    date_col    = row.business_date_field     # e.g. "record_date"
    acct_col    = row.source_account_field    # e.g. "account_number"
    
    full_table = f"retailbank_dev.{table_name}"
    
    # Read source and filter by business date
    df = spark.table(full_table).filter(F.col(date_col) == business_date)
    
    # For INCREMENTAL loads, skip CLOSED accounts
    if load_type == "INCREMENTAL":
        df = df.filter(F.col(stat_col) != "CLOSED")
    
    # Select and rename columns to standard names
    df = df.select(
        F.lit(source_code).alias("source_system_code"),
        F.col(acct_col).alias("source_account_number"),
        F.col(cust_col).alias("customer_id"),
        F.col(prod_col).alias("product_code"),
        F.col(curr_col).alias("currency_code"),
        F.col(bal_col).alias("account_balance"),
        F.col(stat_col).alias("account_status"),
        F.col(date_col).alias("business_date")
    )
    
    source_dataframes.append(df)
    
    if debug:
        print(f"Loaded {source_code}: {df.count()} rows")

# Union all sources
from functools import reduce

portfolio_stage_df = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    source_dataframes
)

rows_read = portfolio_stage_df.count()

if debug:
    print(f"\nTotal portfolio rows read: {rows_read}")
    portfolio_stage_df.groupBy("source_system_code").count().orderBy("source_system_code").show()

Loaded CORE_BANKING: 8 rows
Loaded LOANS: 5 rows
Loaded CARDS: 4 rows
Loaded INVESTMENTS: 3 rows
Loaded MORTGAGE: 3 rows
Loaded MOBILE: 3 rows
Loaded FOREX: 3 rows

Total portfolio rows read: 29
+------------------+-----+
|source_system_code|count|
+------------------+-----+
|             CARDS|    4|
|      CORE_BANKING|    8|
|             FOREX|    3|
|       INVESTMENTS|    3|
|             LOANS|    5|
|            MOBILE|    3|
|          MORTGAGE|    3|
+------------------+-----+



In [0]:
# ============================================================
# DATA CLEANSING
# Same rules as SQL:
#   - Trim spaces, blank -> NULL
#   - Uppercase product codes and currencies
#   - Fix -0.00 balances
# ============================================================

portfolio_stage_df = portfolio_stage_df \
    .withColumn("customer_id",  F.when(F.trim(F.col("customer_id"))  == "", None).otherwise(F.trim(F.col("customer_id")))) \
    .withColumn("product_code", F.upper(F.when(F.trim(F.col("product_code")) == "", None).otherwise(F.trim(F.col("product_code"))))) \
    .withColumn("currency_code", F.upper(F.when(F.trim(F.col("currency_code")) == "", None).otherwise(F.trim(F.col("currency_code"))))) \
    .withColumn("account_status", F.upper(F.col("account_status"))) \
    .withColumn("account_balance", 
        F.when(F.col("account_balance") == -0.00, F.lit(0.00))
         .otherwise(F.col("account_balance"))
    )

if debug:
    print("Cleansing complete.")
    portfolio_stage_df.select("source_system_code", "source_account_number", "customer_id", "product_code", "currency_code", "account_balance", "account_status").show(10, truncate=False)

Cleansing complete.
+------------------+---------------------+-----------+-------------+-------------+---------------+--------------+
|source_system_code|source_account_number|customer_id|product_code |currency_code|account_balance|account_status|
+------------------+---------------------+-----------+-------------+-------------+---------------+--------------+
|CORE_BANKING      |CB10001              |CUST001    |SAV001       |USD          |25000.0        |ACTIVE        |
|CORE_BANKING      |CB10002              |CUST002    |CUR001       |EUR          |75000.0        |ACTIVE        |
|CORE_BANKING      |CB10003              |CUST003    |SAV001       |USD          |0.0            |ACTIVE        |
|CORE_BANKING      |CB10004              |CUST004    |CUR001       |GBP          |-5000.0        |ACTIVE        |
|CORE_BANKING      |CB10005              |NULL       |SAV001       |USD          |10000.0        |ACTIVE        |
|CORE_BANKING      |CB10006              |CUST006    |NULL         |

In [0]:
# ============================================================
# VALIDATION & ERROR CAPTURE
# Check for obvious problems. Each problem goes into
# warehouse.portfolio_errors (the CRITICAL FIX for inconsistency #8).
# These are RAW errors -- just the basic facts. Procedure 4 (nb_04)
# will enrich them later if needed.
# ============================================================

errors_list = []

# Rule 1: Missing Customer
missing_cust = portfolio_stage_df.filter(F.col("customer_id").isNull()).select(
    F.lit(business_date).cast("date").alias("business_date"),
    F.col("source_system_code"),
    F.col("source_account_number"),
    F.lit(None).cast("string").alias("customer_id"),
    F.lit("MISSING_CUSTOMER").alias("error_category"),
    F.lit("Customer identifier is missing.").alias("error_description"),
    F.current_timestamp().alias("logged_date"),
    F.lit(execution_id).alias("execution_id")
)
errors_list.append(missing_cust)

# Rule 2: Missing Product
missing_prod = portfolio_stage_df.filter(F.col("product_code").isNull()).select(
    F.lit(business_date).cast("date").alias("business_date"),
    F.col("source_system_code"),
    F.col("source_account_number"),
    F.col("customer_id"),
    F.lit("MISSING_PRODUCT").alias("error_category"),
    F.lit("Product code is missing.").alias("error_description"),
    F.current_timestamp().alias("logged_date"),
    F.lit(execution_id).alias("execution_id")
)
errors_list.append(missing_prod)

# Rule 3: NULL Balance
null_bal = portfolio_stage_df.filter(F.col("account_balance").isNull()).select(
    F.lit(business_date).cast("date").alias("business_date"),
    F.col("source_system_code"),
    F.col("source_account_number"),
    F.col("customer_id"),
    F.lit("INVALID_BALANCE").alias("error_category"),
    F.lit("Account balance is NULL.").alias("error_description"),
    F.current_timestamp().alias("logged_date"),
    F.lit(execution_id).alias("execution_id")
)
errors_list.append(null_bal)

# Union all errors and write to warehouse.portfolio_errors
if len(errors_list) > 0:
    from functools import reduce
    all_errors_df = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), errors_list)
    all_errors_df = all_errors_df.distinct()
    
    rows_rejected = all_errors_df.count()
    
    if rows_rejected > 0:
        all_errors_df.write.format("delta").mode("append").saveAsTable("retailbank_dev.warehouse.portfolio_errors")
        
        if debug:
            print(f"Validation errors written to PortfolioErrors: {rows_rejected}")
            all_errors_df.show(truncate=False)
else:
    rows_rejected = 0

Validation errors written to PortfolioErrors: 3
+-------------+------------------+---------------------+-----------+----------------+-------------------------------+--------------------------+------------------------------------+
|business_date|source_system_code|source_account_number|customer_id|error_category  |error_description              |logged_date               |execution_id                        |
+-------------+------------------+---------------------+-----------+----------------+-------------------------------+--------------------------+------------------------------------+
|2026-01-31   |CORE_BANKING      |CB10005              |NULL       |MISSING_CUSTOMER|Customer identifier is missing.|2026-08-24 03:25:56.093777|271702c1-6499-48c3-8cc1-ea439ba17853|
|2026-01-31   |CORE_BANKING      |CB10006              |CUST006    |MISSING_PRODUCT |Product code is missing.       |2026-08-24 03:25:56.093777|271702c1-6499-48c3-8cc1-ea439ba17853|
|2026-01-31   |CARDS             |CARD004 

In [0]:
# ============================================================
# CURRENCY CONVERSION
# Join to reference.exchange_rates on currency_code.
# Convert every balance to USD using the rate for the business date.
# If a currency has no rate, flag it as an error.
# ============================================================

# Join FX rates
portfolio_stage_df = portfolio_stage_df.alias("p").join(
    fx_df.alias("fx"),
    (F.col("p.currency_code") == F.col("fx.currency_code")),
    "left"
).select(
    F.col("p.*"),
    F.col("fx.exchange_rate")
)

# Fill missing rates with 1.0 (USD base) and calculate USD balance
portfolio_stage_df = portfolio_stage_df \
    .withColumn("exchange_rate", F.coalesce(F.col("exchange_rate"), F.lit(1.0))) \
    .withColumn("base_currency_balance", F.round(F.col("account_balance") * F.col("exchange_rate"), 2))

# Flag missing exchange rates as errors
missing_fx = portfolio_stage_df.filter(
    (F.col("currency_code").isNotNull()) & (F.col("exchange_rate") == 1.0) & (F.col("currency_code") != "USD")
).select(
    F.lit(business_date).cast("date").alias("business_date"),
    F.col("source_system_code"),
    F.col("source_account_number"),
    F.col("customer_id"),
    F.lit("MISSING_EXCHANGE_RATE").alias("error_category"),
    F.concat(F.lit("Exchange rate not found for currency "), F.col("currency_code")).alias("error_description"),
    F.current_timestamp().alias("logged_date"),
    F.lit(execution_id).alias("execution_id")
)

if missing_fx.count() > 0:
    missing_fx.write.format("delta").mode("append").saveAsTable("retailbank_dev.warehouse.portfolio_errors")
    rows_rejected += missing_fx.count()
    if debug:
        print(f"Missing FX rate errors: {missing_fx.count()}")
        missing_fx.show(truncate=False)

In [0]:
# ============================================================
# ENRICHMENT
# 1. Join to reference.product -> get description, category, regulatory
# 2. Join to warehouse.customer_master -> get name, category, branch
# ============================================================

# 8.1 Product Enrichment
portfolio_stage_df = portfolio_stage_df.alias("p").join(
    product_df.alias("pr"),
    F.col("p.product_code") == F.col("pr.product_code"),
    "left"
).select(
    F.col("p.*"),
    F.col("pr.product_description"),
    F.col("pr.product_category"),
    F.col("pr.regulatory_category")
)

# Flag unknown products
unknown_prod = portfolio_stage_df.filter(F.col("product_description").isNull()).select(
    F.lit(business_date).cast("date").alias("business_date"),
    F.col("source_system_code"),
    F.col("source_account_number"),
    F.col("customer_id"),
    F.lit("UNKNOWN_PRODUCT").alias("error_category"),
    F.concat(F.lit("Unknown Product Code: "), F.col("product_code")).alias("error_description"),
    F.current_timestamp().alias("logged_date"),
    F.lit(execution_id).alias("execution_id")
)

if unknown_prod.count() > 0:
    unknown_prod.write.format("delta").mode("append").saveAsTable("retailbank_dev.warehouse.portfolio_errors")
    rows_rejected += unknown_prod.count()
    if debug:
        print(f"Unknown product errors: {unknown_prod.count()}")

# 8.2 Customer Master Enrichment
portfolio_stage_df = portfolio_stage_df.alias("p").join(
    customer_df.alias("c"),
    F.col("p.customer_id") == F.col("c.customer_id"),
    "left"
).select(
    F.col("p.*"),
    F.concat(F.col("c.first_name"), F.lit(" "), F.col("c.last_name")).alias("customer_name"),
    F.col("c.customer_category").alias("customer_master_category"),
    F.col("c.branch_code")
)

# Flag customers not found in master
missing_cust_master = portfolio_stage_df.filter(
    (F.col("customer_id").isNotNull()) & (F.col("customer_name").isNull())
).select(
    F.lit(business_date).cast("date").alias("business_date"),
    F.col("source_system_code"),
    F.col("source_account_number"),
    F.col("customer_id"),
    F.lit("CUSTOMER_NOT_FOUND").alias("error_category"),
    F.lit("Customer not found in Customer Master.").alias("error_description"),
    F.current_timestamp().alias("logged_date"),
    F.lit(execution_id).alias("execution_id")
)

if missing_cust_master.count() > 0:
    missing_cust_master.write.format("delta").mode("append").saveAsTable("retailbank_dev.warehouse.portfolio_errors")
    rows_rejected += missing_cust_master.count()
    if debug:
        print(f"Customer not found errors: {missing_cust_master.count()}")

if debug:
    print("Enrichment complete.")
    portfolio_stage_df.select("source_account_number", "customer_id", "customer_name", "product_description", "base_currency_balance").show(10, truncate=False)

Unknown product errors: 6
Customer not found errors: 7
Enrichment complete.
+---------------------+-----------+----------------+-------------------------+---------------------+
|source_account_number|customer_id|customer_name   |product_description      |base_currency_balance|
+---------------------+-----------+----------------+-------------------------+---------------------+
|CB10001              |CUST001    |Customer CUST001|Savings Account          |25000.0              |
|CB10002              |CUST002    |Customer CUST002|Current Account          |81000.0              |
|CB10003              |CUST003    |Customer CUST003|Savings Account          |0.0                  |
|CB10004              |CUST004    |Customer CUST004|Current Account          |-6250.0              |
|CB10005              |NULL       |NULL            |Savings Account          |10000.0              |
|CB10006              |CUST006    |Customer CUST006|NULL                     |15000.0              |
|CB10007       

In [0]:
# ============================================================
# BUSINESS RULES FILTER
# Exclude accounts that should NOT appear in the golden data:
#   - CLOSED status
#   - Zero balance
#   - Unknown product (NULL description)
#   - Missing exchange rate (NULL rate for non-USD)
#   - Customers with OPEN CRITICAL exceptions in CustomerMasterExceptions
#   - NULL customer_id (FIX #8: these cannot be matched by MERGE)
# ============================================================

# Build the exclusion flag
portfolio_stage_df = portfolio_stage_df.withColumn(
    "include_portfolio",
    F.when(F.col("account_status") == "CLOSED", 0)
     .when(F.col("base_currency_balance") == 0, 0)
     .when(F.col("product_description").isNull(), 0)
     .when((F.col("exchange_rate").isNull()) & (F.col("currency_code") != "USD"), 0)
     .when(F.col("customer_id").isNull(), 0)    # FIX #8: NULL CustomerId breaks MERGE
     .otherwise(1)
)

# Exclude customers with OPEN CRITICAL exceptions
# LEFT SEMI JOIN: keep rows where customer_id EXISTS in critical_exceptions
portfolio_stage_df = portfolio_stage_df.alias("p").join(
    critical_exceptions_df.alias("ce"),
    F.col("p.customer_id") == F.col("ce.customer_id"),
    "left_anti" 
).select("p.*")

# Final filtered set
enterprise_df = portfolio_stage_df.filter(F.col("include_portfolio") == 1).drop("include_portfolio")

if debug:
    print(f"Rows after business rules: {enterprise_df.count()}")
    enterprise_df.select("source_system_code", "source_account_number", "customer_id", "base_currency_balance").show(10, truncate=False)

Rows after business rules: 19
+------------------+---------------------+-----------+---------------------+
|source_system_code|source_account_number|customer_id|base_currency_balance|
+------------------+---------------------+-----------+---------------------+
|CORE_BANKING      |CB10001              |CUST001    |25000.0              |
|CORE_BANKING      |CB10002              |CUST002    |81000.0              |
|CORE_BANKING      |CB10004              |CUST004    |-6250.0              |
|CORE_BANKING      |CB10007              |CUST007    |45000.0              |
|LOANS             |LN10001              |CUST001    |500000.0             |
|LOANS             |LN10002              |CUST002    |27000.0              |
|LOANS             |LN10003              |CUST009    |937500.0             |
|LOANS             |LN10004              |CUST010    |-1000.0              |
|CARDS             |CARD001              |CUST001    |12000.0              |
|CARDS             |CARD002              |CUST

In [0]:
# ============================================================
# PORTFOLIO METRICS
#   - PortfolioValueBand: STANDARD/SILVER/GOLD/PLATINUM
#   - HighValueCustomer: > $500k
#   - ProductCount: how many products per customer
# ============================================================

# 10.1 Value Band
enterprise_df = enterprise_df.withColumn(
    "portfolio_value_band",
    F.when(F.col("base_currency_balance") >= 1000000, "PLATINUM")
     .when(F.col("base_currency_balance") >= 250000,  "GOLD")
     .when(F.col("base_currency_balance") >= 50000,   "SILVER")
     .otherwise("STANDARD")
)

# 10.2 High Value Flag
enterprise_df = enterprise_df.withColumn(
    "high_value_customer",
    F.when(F.col("base_currency_balance") >= 500000, True).otherwise(False)
)

# 10.3 Eligible for Reporting (default Y unless excluded)
enterprise_df = enterprise_df.withColumn(
    "eligible_for_reporting",
    F.lit("Y")
)

# 10.4 Product Count per Customer
# We need to count products per customer BEFORE we add the column,
# because we can't reference a window function in the same select easily.
# Instead, we'll do a two-step: calculate count, then join back.

from pyspark.sql.window import Window
cust_window = Window.partitionBy("customer_id")

enterprise_df = enterprise_df.withColumn(
    "product_count",
    F.count("*").over(cust_window)
)

if debug:
    print("Portfolio metrics calculated.")
    enterprise_df.select("customer_id", "base_currency_balance", "portfolio_value_band", "high_value_customer", "product_count").show(10, truncate=False)

Portfolio metrics calculated.
+-----------+---------------------+--------------------+-------------------+-------------+
|customer_id|base_currency_balance|portfolio_value_band|high_value_customer|product_count|
+-----------+---------------------+--------------------+-------------------+-------------+
|CUST001    |25000.0              |STANDARD            |false              |6            |
|CUST001    |500000.0             |GOLD                |true               |6            |
|CUST001    |12000.0              |STANDARD            |false              |6            |
|CUST001    |150000.0             |SILVER              |false              |6            |
|CUST001    |500.0                |STANDARD            |false              |6            |
|CUST001    |250000.0             |GOLD                |false              |6            |
|CUST002    |81000.0              |SILVER              |false              |2            |
|CUST002    |27000.0              |STANDARD            |fals

In [0]:
# ============================================================
# FINAL VALIDATION
# Flag negative balances below -10,000 as errors.
# These stay in the portfolio but are logged.
# ============================================================

negative_bal = enterprise_df.filter(F.col("base_currency_balance") < -10000).select(
    F.lit(business_date).cast("date").alias("business_date"),
    F.col("source_system_code"),
    F.col("source_account_number"),
    F.col("customer_id"),
    F.lit("NEGATIVE_PORTFOLIO").alias("error_category"),
    F.lit("Portfolio balance below acceptable threshold.").alias("error_description"),
    F.current_timestamp().alias("logged_date"),
    F.lit(execution_id).alias("execution_id")
)

if negative_bal.count() > 0:
    negative_bal.write.format("delta").mode("append").saveAsTable("retailbank_dev.warehouse.portfolio_errors")
    rows_rejected += negative_bal.count()
    if debug:
        print(f"Negative balance errors: {negative_bal.count()}")

Negative balance errors: 1


In [0]:
# ============================================================
# PREPARE TARGET COLUMNS
# Select and rename columns to match warehouse.customer_portfolio exactly.
# ============================================================

final_df = enterprise_df.select(
    F.col("business_date"),
    F.col("source_system_code"),
    F.col("source_account_number"),
    F.col("customer_id"),
    F.col("customer_name"),
    F.col("customer_master_category").alias("customer_category"),
    F.col("branch_code"),
    F.col("product_code"),
    F.col("product_description"),
    F.col("product_category"),
    F.col("regulatory_category"),
    F.col("currency_code"),
    F.col("exchange_rate"),
    F.col("account_balance"),
    F.col("base_currency_balance"),
    F.col("account_status"),
    F.col("eligible_for_reporting"),
    F.col("portfolio_value_band"),
    F.col("high_value_customer"),
    F.col("product_count"),
    F.current_timestamp().alias("created_date"),
    F.current_timestamp().alias("last_updated_date")
)

if debug:
    print(f"Final rows ready for MERGE: {final_df.count()}")
    final_df.printSchema()

Final rows ready for MERGE: 19
root
 |-- business_date: date (nullable = true)
 |-- source_system_code: string (nullable = false)
 |-- source_account_number: string (nullable = false)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_category: string (nullable = true)
 |-- branch_code: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_description: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- regulatory_category: string (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- exchange_rate: double (nullable = false)
 |-- account_balance: double (nullable = true)
 |-- base_currency_balance: double (nullable = true)
 |-- account_status: string (nullable = true)
 |-- eligible_for_reporting: string (nullable = false)
 |-- portfolio_value_band: string (nullable = false)
 |-- high_value_customer: boolean (nullable = false)
 |-- product_count: long (nullable = false)


In [0]:
# ============================================================
# MERGE INTO WAREHOUSE.CUSTOMER_PORTFOLIO
# Match on: BusinessDate + CustomerId + SourceAccountNumber
# This prevents the same account appearing twice for the same
# customer on the same day (same as the unique index in SQL).
# ============================================================

target_table = "retailbank_dev.warehouse.customer_portfolio"
delta_target = DeltaTable.forName(spark, target_table)

if final_df.count() > 0:
    
    delta_target.alias("target").merge(
        final_df.alias("source"),
        """
        target.business_date = source.business_date 
        AND target.customer_id = source.customer_id 
        AND target.source_account_number = source.source_account_number
        """
    ).whenMatchedUpdate(
        condition="""
            COALESCE(target.product_code,'')          <> COALESCE(source.product_code,'') OR
            COALESCE(target.product_category,'')      <> COALESCE(source.product_category,'') OR
            COALESCE(target.currency_code,'')         <> COALESCE(source.currency_code,'') OR
            COALESCE(target.exchange_rate,0)          <> COALESCE(source.exchange_rate,0) OR
            COALESCE(target.account_balance,0)        <> COALESCE(source.account_balance,0) OR
            COALESCE(target.base_currency_balance,0)  <> COALESCE(source.base_currency_balance,0) OR
            COALESCE(target.account_status,'')        <> COALESCE(source.account_status,'') OR
            COALESCE(target.portfolio_value_band,'')  <> COALESCE(source.portfolio_value_band,'') OR
            COALESCE(target.high_value_customer,False)<> COALESCE(source.high_value_customer,False) OR
            COALESCE(target.product_count,0)          <> COALESCE(source.product_count,0)
        """,
        set={
            "customer_name":         "source.customer_name",
            "customer_category":     "source.customer_category",
            "branch_code":           "source.branch_code",
            "product_code":          "source.product_code",
            "product_description":   "source.product_description",
            "product_category":      "source.product_category",
            "regulatory_category":   "source.regulatory_category",
            "currency_code":         "source.currency_code",
            "exchange_rate":         "source.exchange_rate",
            "account_balance":       "source.account_balance",
            "base_currency_balance": "source.base_currency_balance",
            "account_status":        "source.account_status",
            "eligible_for_reporting":"source.eligible_for_reporting",
            "portfolio_value_band":  "source.portfolio_value_band",
            "high_value_customer":   "source.high_value_customer",
            "product_count":         "source.product_count",
            "last_updated_date":     "source.last_updated_date"
        }
    ).whenNotMatchedInsert(
        values={
            "business_date":         "source.business_date",
            "source_system_code":    "source.source_system_code",
            "source_account_number": "source.source_account_number",
            "customer_id":           "source.customer_id",
            "customer_name":         "source.customer_name",
            "customer_category":     "source.customer_category",
            "branch_code":           "source.branch_code",
            "product_code":          "source.product_code",
            "product_description":   "source.product_description",
            "product_category":      "source.product_category",
            "regulatory_category":   "source.regulatory_category",
            "currency_code":         "source.currency_code",
            "exchange_rate":         "source.exchange_rate",
            "account_balance":       "source.account_balance",
            "base_currency_balance": "source.base_currency_balance",
            "account_status":        "source.account_status",
            "eligible_for_reporting":"source.eligible_for_reporting",
            "portfolio_value_band":  "source.portfolio_value_band",
            "high_value_customer":   "source.high_value_customer",
            "product_count":         "source.product_count",
            "created_date":          "source.created_date",
            "last_updated_date":     "source.last_updated_date"
        }
    ).execute()

    # Capture MERGE stats
    history_df = spark.sql(f"DESCRIBE HISTORY {target_table}")
    latest_merge = history_df.filter("operation = 'MERGE'").orderBy(F.desc("version")).limit(1)
    
    if latest_merge.count() > 0:
        metrics = latest_merge.select("operationMetrics").collect()[0][0]
        rows_inserted = int(metrics.get("numTargetRowsInserted", "0"))
        rows_updated  = int(metrics.get("numTargetRowsUpdated", "0"))
    else:
        rows_inserted = 0
        rows_updated  = 0
else:
    rows_inserted = 0
    rows_updated  = 0
    if debug:
        print("No rows to MERGE.")

if debug:
    print(f"MERGE complete: {rows_inserted} inserted, {rows_updated} updated")

MERGE complete: 0 inserted, 0 updated


In [0]:
# ============================================================
# AUDIT: PORTFOLIO EXECUTION SUMMARY
# Daily statistics for management dashboards.
# ============================================================

end_time         = datetime.now()
duration_seconds = int((end_time - start_time).total_seconds())

# Calculate summary stats from final_df
if final_df.count() > 0:
    summary = final_df.groupBy().agg(
        F.countDistinct("customer_id").alias("total_customers"),
        F.count("*").alias("total_accounts"),
        F.sum("base_currency_balance").alias("total_portfolio_value")
    ).collect()[0]
    
    total_cust = summary.total_customers or 0
    total_acct = summary.total_accounts or 0
    total_val  = summary.total_portfolio_value or 0
else:
    total_cust = 0
    total_acct = 0
    total_val  = 0

spark.sql(f"""
    INSERT INTO retailbank_dev.audit.portfolio_execution_summary
    (business_date, procedure_name, execution_id, total_customers, total_accounts, total_portfolio_value, total_errors, execution_time_seconds, created_date)
    VALUES
    ('{business_date}', '{procedure_name}', '{execution_id}', {total_cust}, {total_acct}, {total_val}, {rows_rejected}, {duration_seconds}, '{end_time.strftime("%Y-%m-%d %H:%M:%S")}')
""")

# Final audit update
status  = "SUCCESS"
message = (
    f"Customer Portfolio Load Completed Successfully. "
    f"Rows Read: {rows_read}, Rows Loaded: {rows_inserted}, "
    f"Rows Updated: {rows_updated}, Rows Rejected: {rows_rejected}"
)

write_audit_end(status, message)

if debug:
    print("=" * 50)
    print("CUSTOMER PORTFOLIO LOAD SUMMARY")
    print("=" * 50)
    print(f"Rows Read        : {rows_read}")
    print(f"Rows Loaded      : {rows_inserted}")
    print(f"Rows Updated     : {rows_updated}")
    print(f"Rows Rejected    : {rows_rejected}")
    print(f"Total Customers  : {total_cust}")
    print(f"Total Accounts   : {total_acct}")
    print(f"Portfolio Value  : {total_val}")
    print(f"Status           : {status}")
    print("=" * 50)
    
    if final_df.count() > 0:
        final_df.groupBy("product_category").count().orderBy(F.desc("count")).show()

print("nb_03_CustomerPortfolio completed successfully.")

CUSTOMER PORTFOLIO LOAD SUMMARY
Rows Read        : 29
Rows Loaded      : 0
Rows Updated     : 0
Rows Rejected    : 17
Total Customers  : 12
Total Accounts   : 19
Portfolio Value  : 3261330.0
Status           : SUCCESS
+----------------+-----+
|product_category|count|
+----------------+-----+
|         Deposit|    4|
|           Loans|    4|
|           Cards|    4|
|     Investments|    3|
|         Digital|    2|
|           Forex|    2|
+----------------+-----+

nb_03_CustomerPortfolio completed successfully.
